<a href="https://colab.research.google.com/github/MParvan/ecg-biometrics-bench/blob/main/Custom_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Benchmarking Your Own Custom Dataset
While the ECG-Biometrics-Bench framework comes with automated downloaders for 7 major public datasets, you will likely want to evaluate your own private or clinical data.

Because the core engine (run.py) relies entirely on standard NumPy arrays, bypassing the built-in dataset loaders is incredibly straightforward.

# Step 1: Prepare Your Data
The framework only needs two things to run a biometric evaluation:

X: A NumPy array of your ECG signals. Shape: (N_samples, Time_Length)

y: A NumPy array of your subject IDs/labels. Shape: (N_samples,)

In [ ]:
import numpy as np
import pandas as pd
from preprocessing import Preprocessing
from run import run_closed_set_identification

# Let's simulate loading your private local dataset
# Assume you have a folder of CSVs where each file is a 10-second ECG recording
my_fs = 250 # 250 Hz sampling rate

# 1. Initialize the framework's preprocessor
# This gives your custom data the exact same filtering/normalization as the public datasets!
preprocessor = Preprocessing()

X_custom = []
y_custom = []

# --- DUMMY DATA CREATION (Replace this with your actual CSV loading logic) ---
# Simulating 3 subjects, each with 20 ECG recordings of 10 seconds
subject_ids = ['Patient_A', 'Patient_B', 'Patient_C']
for subj in subject_ids:
    for _ in range(20):
        # Simulating a raw noisy signal: 10 seconds * 250 Hz = 2500 samples
        raw_signal = np.random.randn(2500)

        # 2. Preprocess your raw signal using the framework's engine
        segments = preprocessor.preprocess_ecg(
            raw_signal, fs=my_fs,
            mode='blind',       # Use blind windowing (no peak detection needed)
            window_s=5.0,       # Cut into 5-second chunks
            stride_s=2.5,       # 50% overlap
            filter_method='butter',
            norm_method='zscore'
        )

        # Append the cleaned segments
        if len(segments) > 0:
            X_custom.append(segments)
            y_custom.extend([subj] * len(segments))

# 3. Stack into the required framework formats
X_custom = np.vstack(X_custom)
y_custom = np.array(y_custom)

print(f"Prepared Custom Dataset: {X_custom.shape[0]} segments across {len(np.unique(y_custom))} subjects.")
print(f"Input dimension for neural net: {X_custom.shape[1]}")

# Step 2: Feed into the Biometric Engine
Now that you have X and y, you can pass them directly into any of the 8 biometric tasks in run.py.

In [ ]:
print("\n--- Benchmarking Custom Data ---")

# Run Task 1 (Closed-Set Identification) directly on your local data
metrics, stats, hyperparams = run_closed_set_identification(
    X_custom,
    y_custom,
    model_name='deepecg',
    epochs=30,
    batch_size=32,
    test_split=0.2
)

rank1, rank5 = metrics
print(f"\nCustom Data Results | Rank-1 Accuracy: {rank1*100:.2f}%")